In [5]:
import torch
from pathlib import Path
from torch_geometric.nn import to_hetero

from halide_gnn_cost_model.data import PipelineDataset
from halide_gnn_cost_model.model import PipeGCN, PipeGAT, PipelineModel

In [2]:
PIPELINES_DIR = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/pipelines-test")
GCN_MODEL_PATH = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/models/gcn/pipeline_model_epoch_100.pt")
GAT_MODEL_PATH = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/models/gat/pipeline_model_epoch_100.pt")

In [3]:
USE_GPU = True

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_GPU and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

Using device: mps


In [4]:
# Load dataset
dataset = PipelineDataset(PIPELINES_DIR)
data = dataset[0]  # Get the first pipeline graph
len(dataset)

1lines [00:00, 2000.14lines/s]
1lines [00:00, 35848.75lines/s]


2157

# GCN Eval

In [7]:
DIM_EMBEDDING = 64
gcn = PipeGCN(hidden_channels=DIM_EMBEDDING, out_channels=DIM_EMBEDDING, num_layers=4)
gcn = to_hetero(gcn, data.metadata(), aggr="sum")
model = PipelineModel(gcn, DIM_EMBEDDING, 5, len(dataset.ast_vocab), len(dataset.sched_vocab))
model.load_state_dict(torch.load(GCN_MODEL_PATH, map_location=device))

/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'drop

<All keys matched successfully>

## Avergate Runtime Error

In [ ]:
DATA_IDX = 4  # Index of the data point to evaluate
torch.exp(model(dataset[DATA_IDX])), dataset[DATA_IDX].y

(tensor([[   3.9573,   17.2393,   74.7610,  349.0638, 1430.9464]],
        grad_fn=<ExpBackward0>),
 tensor([   2.5992,   11.1184,   55.4568,  262.8322, 1987.4459]))

In [21]:
def average_runtime_error(model, dataset):
    model.eval()
    total_error = 0
    with torch.no_grad():
        for graph in dataset:
            pred_log = model(graph)
            pred = torch.exp(pred_log)
            true = graph.y
            error = torch.abs(pred - true) / true
            total_error += error.mean().item()
    return total_error / len(dataset)

In [22]:
print(f"Average Runtime Error on Test Set: {average_runtime_error(model, dataset) * 100:.2f}%")

Average Runtime Error on Test Set: 52.28%
